# Lab 4: Navier–Stokes flow

[Start Here](../../Start_Here.ipynb) · Previous: [Lab 3: Heat conduction](../03_heat_conduction/Lab_3_Heat_Conduction.ipynb) · Next: [Challenge 1: Wave](../../02_challenges/01_wave/Challenge_1_Wave_Dynamics.ipynb)

Start with the original bootcamp's wind and pressure array, described upstream as projected and tiled ERA5 data. Train a PINN on that initial field and the Navier–Stokes equations, then watch its predicted flow evolve over 60 hours.

First inspect the input wind, then play the 11 predicted frames in this notebook or ParaView. Colors show speed; arrows show flow direction and magnitude.

The input is $(x,y,t)$ and the outputs are two velocity components and pressure. Initial data and the three PDE residuals train the model; periodic coordinate features enforce spatial periodicity.

This is a simplified fluid-flow experiment, not a validated weather forecast. It omits thermodynamics, moisture, planetary rotation, and spherical geometry; no future weather observations are supplied.

The playback uses one fixed speed scale across all times. Coordinates, velocity and pressure stay in the original lesson's normalized units.

## The flow problem

The network predicts horizontal velocity $u$, vertical velocity $v$, and pressure $p$. With constant density and viscosity, the equations are

$$u_x+v_y=0,$$
$$u_t+uu_x+vu_y+p_x/\rho-\nu(u_{xx}+u_{yy})=0,$$
$$v_t+uv_x+vv_y+p_y/\rho-\nu(v_{xx}+v_{yy})=0.$$

The first equation enforces incompressibility; the other two balance momentum.

### The original initial condition

The supplied `data_lat.npy` provides the initial u, v, and pressure fields. We retain the original array and its normalization. Its acquisition date and preprocessing metadata are incomplete; see [data provenance](DATA_PROVENANCE.md). The images below explain the projection and tiling used by the original lesson.

![Projection illustration from the original lesson](images/projection.png)

### Background: general conservation equations

The general equations below also allow variable density and energy transport. This Lab trains on only the constant-density continuity and momentum equations above. The energy equation is background.

\begin{equation}
Continuity : \frac{\partial \rho}{\partial t} + \overrightarrow{\nabla}\cdot(\rho\overrightarrow{u})=0 \end{equation}

\begin{equation}
Momentum : \frac{\partial(\rho \overrightarrow{u})}{\partial t} + \overrightarrow{\nabla}\cdot[\rho\overline{\overline{u\otimes u}}] = -\overrightarrow{\nabla p} + \overrightarrow{\nabla}\cdot\overline{\overline{\tau}} + \rho\overrightarrow{f} \end{equation}

\begin{equation}
Energy : \frac{\partial(\rho e)}{\partial t} + \overrightarrow{\nabla}\cdot((\rho e + p)\overrightarrow{u}) = \overrightarrow{\nabla}\cdot(\overline{\overline{\tau}}\cdot\overrightarrow{u}) + \rho\overrightarrow{f}\overrightarrow{u} + \overrightarrow{\nabla}\cdot(\overrightarrow{\dot{q}})+r \end{equation}

### Step 1: Periodic domain

The domain is $x,y\in[-0.720,0.720]$. Sine and cosine input features make the network periodic in both directions. The original array supplies tiled samples from -0.720 to 0.719.

![Periodic tiling illustration](images/periodicity_conversion.png)

These planar periodic boundaries are a modeling choice, not a representation of a spherical atmosphere.

### Scaling and nondimensionalization

```python
LENGTH = 1.440
LOWER = -0.720
LENGTH_SCALE = 12742000 / LENGTH
TIME_SCALE = 60 * 60 * 60
VELOCITY_SCALE = LENGTH_SCALE / TIME_SCALE
PRESSURE_SCALE = 1.1614 * VELOCITY_SCALE**2
LEGACY_PRESSURE_FACTOR = 0.10197
REAL_NU = 1.655e-5 / (LENGTH_SCALE**2 / TIME_SCALE)
```

```python
def read_wf_data(velocity_scale=VELOCITY_SCALE, pressure_scale=PRESSURE_SCALE, data_path=None):
    """Load tiled coordinates and fields, including the
    unvalidated 0.10197 pressure factor. See DATA_PROVENANCE.md before interpreting units.
    """
    path = Path(data_path) if data_path else Path(__file__).resolve().parents[1] / "data_lat.npy"
    if not path.is_file():
        raise FileNotFoundError(f"Missing original data: {path}. Restore data_lat.npy before running this Lab.")
    ic = np.load(path, allow_pickle=False).astype(np.float32)
    if ic.ndim != 3 or ic.shape[0] != 3 or not np.isfinite(ic).all():
        raise ValueError("Expected finite upstream data shaped (3, H, W) for u, v, p")
    mesh_y, mesh_x = np.meshgrid(np.linspace(-.720, .719, ic.shape[1]),
                                np.linspace(-.720, .719, ic.shape[2]), indexing="ij")
    xy = np.column_stack((mesh_x.ravel(), mesh_y.ravel())).astype(np.float32)
    fields = np.column_stack((ic[0].ravel() / velocity_scale,
                              ic[1].ravel() / velocity_scale,
                              ic[2].ravel() * LEGACY_PRESSURE_FACTOR / pressure_scale))
    return xy, fields.astype(np.float32)
```

The loader multiplies the array's pressure channel by `0.10197` before scaling it. The units and pressure offset behind this factor are unverified; it is not a valid conversion from pressure to density. Treat this data path as a numerical example until its units are established.

### Step 2: Equations and periodic network

```python
# Use the local equation class defined below.
physics = informer(NavierStokes(nu=REAL_NU, rho=1.0, dim=2, time=True), device,
                   supplied_derivatives=("u__t", "v__t"))
```

```python
# Forward calculation inside PeriodicFlow for the class preset.
# bands = [1, 2, 4, 8]; center and scale come from training data only.
phase = 2 * math.pi * (xy - LOWER) / LENGTH
phase = (phase[:, :, None] * self.bands).flatten(1)
features = torch.cat((phase.sin(), phase.cos(), t), dim=1)  # 17 inputs
values = self.network(features)
prediction = self.center + self.scale * values
```

The integer frequencies describe both broad and finer spatial patterns while keeping opposite edges periodic. Output centering and scaling help the optimizer handle velocity and pressure together. The final line restores the original normalized u/v/p before their derivatives enter the equations. The input data and PDE are unchanged.

```python
class NavierStokes(PDE):
    """Constant-density, 2-D incompressible flow equations."""
    def __init__(self, nu, rho=1.0, dim=2, time=True):
        if dim != 2 or not time:
            raise ValueError("This Lab defines the unsteady 2-D equations only")
        self.dim = 2
        x, y, t = Symbol("x"), Symbol("y"), Symbol("t")
        u, v, p = (Function(name)(x, y, t) for name in ("u", "v", "p"))
        self.equations = {
            "continuity": u.diff(x) + v.diff(y),
            "momentum_x": u.diff(t) + u * u.diff(x) + v * u.diff(y)
                          + p.diff(x) / rho - nu * (u.diff(x, 2) + u.diff(y, 2)),
            "momentum_y": v.diff(t) + u * v.diff(x) + v * v.diff(y)
                          + p.diff(y) / rho - nu * (v.diff(x, 2) + v.diff(y, 2)),
        }
```


### Step 3: Initial data and physics loss

The initial loss compares predicted u, v, and p with the data at t=0. The interior loss penalizes continuity and momentum residuals. The source lesson's observation-sum and interior-area loss scaling are retained. Autograd computes u_t and v_t; PhysicsInformer receives these and computes x/y derivatives from the coordinate tensor. The periodic input features make opposite boundaries match. No positive-time reference values train the network.

```python
def residuals(field, xy, t, physics):
    u, v, p = field.split(1, dim=1)
    return physics.forward({"coordinates": xy, "u": u, "v": v, "p": p,
                            "u__t": derivative(u, t), "v__t": derivative(v, t)})


def original_loss_terms(model, physics, points):
    raw_xy, raw_t, ix, target = points
    xy, t = raw_xy.detach().requires_grad_(), raw_t.detach().requires_grad_()
    res = residuals(model(xy, t), xy, t, physics)
    prediction = model(ix, torch.zeros_like(ix[:, :1]))
    return {"physics": LENGTH**2 * sum(v.square().mean() for v in res.values()),
            "initial_data": (prediction - target).square().sum()}
```

The class preset uses FP32 with six 256-unit hidden layers, SiLU activations and weight normalization. Adam starts at learning rate 0.001 and decays by a factor of 0.95 every 3,000 updates. The default budget is 3,000 updates, with 2,048 samples per constraint. A 32-by-32 initial grid is excluded from training and from calculation of the output mean and scale. The remaining source observations train the model.

This preset was selected for class time and initial-field fit, not certified convergence. It uses four periodic feature frequencies instead of one. The original 50,000-update settings remain available with `--recipe upstream`; the original data, equations and loss weights are retained in both recipes. See the [measured comparison](../../ETC/course_materials/LAB4_EFFICIENCY.md).

### Step 4: Evaluate the flow

Compare the original input with the prediction at t=0 to inspect the initial-data fit. At later times, inspect continuity and momentum residuals on held-out points and the matching periodic edges. These are checks of the learned equations and conditions, not comparisons with future weather observations.

The supplied wind has substantial discrete divergence in this planar periodic representation. Fitting it more closely can therefore increase the continuity residual near t=0. Inspect initial-data error and the per-time PDE residuals separately. A small residual later in time can also accompany an overly smooth, weakened flow; that alone is not a successful forecast. The input array is not silently projected or replaced.

Pressure is determined only up to a spatial constant that can vary with time. Pressure gradients drive the flow; an overall pressure offset is not itself a change in velocity.

Normalized time 1 corresponds to 60 hours in the original scaling. The output contains 11 frames at six-hour intervals. They are evaluations of the trained model, not eleven separate training runs.

### Step 5: Inspect the original wind field

Run the setup and input-view cells below. This notebook always uses the supplied `data_lat.npy`; a missing file stops execution. Training starts a new model. No pretrained model is bundled.

[Training program](source_code/navier_stokes.py) · [Configuration](source_code/conf/config.yaml) · [Supplied array](data_lat.npy)

In [ ]:
import os
import sys
import subprocess
import uuid
from pathlib import Path
import numpy as np
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Start_Here.ipynb").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the bootcamp repository.")
sys.path.insert(0, str(ROOT))
from ETC.runtime.notebook import show_results, validate_settings, completed_output
from ETC.runtime.flow_visualization import plot_initial_flow, load_original_flow, animate_flow
from IPython.display import display, FileLink
LAB = ROOT / "01_labs/04_navier_stokes"
OUTPUT_BASE = Path(os.environ.get("AI4SCI_OUTPUT_DIR", str(LAB / "outputs"))).expanduser().resolve()
DEVICE = os.environ.get("AI4SCI_DEVICE", "cpu")
STEPS = int(os.environ.get("AI4SCI_STEPS", "3000"))  # Measured FP32 class preset; initial fit and PDE errors remain separate
RUN_DIRS = {}
RUN_COMPLETED = {}
validate_settings(DEVICE, STEPS)
print("PhysicsNeMo target: 2.2.2", "device:", DEVICE, "steps:", STEPS)

In [ ]:
from importlib import import_module
read_wf_data = import_module("01_labs.04_navier_stokes.source_code.navier_stokes").read_wf_data
initial_xy, initial_fields = read_wf_data(data_path=LAB / "data_lat.npy")
print("Input: original data_lat.npy | normalized u, v, p | initial time: 0 hours")
plot_initial_flow(initial_xy, initial_fields)
plt.show()

### Watch the original ParaView demonstration

Play the original tutorial's recording to see the flow visualization before starting the longer training run. **This is a recording from the original tutorial, not the output of this run.** The model you train below produces its own separately labeled playback and ParaView export.

In [ ]:
from IPython.display import Video
display(Video(str(LAB / "images/paraview.webm"), embed=True, width=850))

### Step 6: Train the flow model

Run the training cell, then inspect the initial-data fit and the PDE residuals. `data_kind` must be `upstream_data_lat_legacy_normalization`.

`initial_data` in `loss.csv` is the sum of initial-field squared errors over the sampled observations. `heldout_after.initial_data_rmse` is an unweighted fit error on a separate spatial grid; `heldout_after.pde_rmse` checks the equations on held-out points. These are different quantities, so do not compare their raw magnitudes as if they were the same error.

There are no future-weather targets for this array, so later frames have no reference/error panels. `weather_forecast_validated` is `False`. The 3,000-step class preset uses FP32; a completed run or falling loss alone does not establish physical accuracy. Initial validation uses reserved observations; PDE validation uses a 64-by-64 spatial grid at six times. Compare the initial wind and predicted frame 0 before interpreting later frames. See [data provenance](DATA_PROVENANCE.md) for the array's unit limitations.

In [ ]:
RUN_COMPLETED["navier_stokes"] = False
OUTPUT = OUTPUT_BASE / ("navier_stokes-" + uuid.uuid4().hex[:8])
RUN_DIRS["navier_stokes"] = OUTPUT
command = [sys.executable, str(LAB / "source_code/navier_stokes.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)]
validate_settings(DEVICE, STEPS)
subprocess.run(command, check=True, cwd=ROOT)
flow = load_original_flow(OUTPUT)
RUN_COMPLETED["navier_stokes"] = True
metrics = show_results(OUTPUT, steps=STEPS, seed=42, preview=False)

### Play the flow: 0 to 60 hours

Press **Play** or drag the frame slider. The left panel stays on the original input at 0 hours; the right panel advances through the PINN predictions. At frame 0, compare how well the model captured the input. Then follow changing flow directions and speed over the next 60 hours.

Color and arrow scales stay fixed across all 11 frames. The left panel is not a future reference, and the axes are normalized planar coordinates, not a latitude/longitude map. The animation uses saved predictions and does not retrain the model.

In [ ]:
OUTPUT = completed_output(RUN_DIRS, RUN_COMPLETED, "navier_stokes")
flow = load_original_flow(OUTPUT)
display(animate_flow(flow))

## View the flow in ParaView

The next cell exports all 11 frames as VTI files and a `flow.pvd` time-series index. Download the ZIP using the link below and extract it on your Mac. This export uses saved predictions; it does not train again.

1. In ParaView, choose **File → Open**, select the extracted `flow.pvd`, then **Apply**.
2. Choose **Surface**, color by **speed**, and look along **+Z**. Use **Reset Camera** if needed.
3. Use **Rescale to Data Range Over All Timesteps** once so colors keep the same meaning while the animation plays.
4. Press **Play** to advance from 0 to 60 hours. For flow arrows, add **Glyph**, select the `velocity` vector, and use the same vector for orientation and scale.

The ZIP includes a README and an optional `view_flow.py` for a prepared view. Field values remain normalized; the time labels use the original 60-hour scaling. The [original demonstration video](images/paraview.webm) is historical material, not a prediction generated by your current run.

In [ ]:
OUTPUT = completed_output(RUN_DIRS, RUN_COMPLETED, "navier_stokes")
from ETC.runtime.paraview import export_paraview
load_original_flow(OUTPUT)  # Do not export an older synthetic result as this Lab.
export = export_paraview(OUTPUT)
print("Download and extract the ZIP, then open flow.pvd in ParaView:")
display(FileLink(os.path.relpath(export["archive"], Path.cwd())))

### Next: build the problem yourself

The Labs provided complete programs. In [Wave Level 1](../../02_challenges/01_wave/wave_l1.py), complete `student_equations`, `student_speed`, and `student_conditions`, save the Python file, and run the notebook cell. Later levels add tasks such as obstacle geometry, coupled parameters, and reference solutions. Each level's required functions are listed in the [challenge task and submission guide](../../ETC/course_materials/CHALLENGE_CONTRACTS.md). Compare the PDE, initial-condition, and boundary-condition errors with the plot. An unfinished function identifies the missing exercise.

Use `USE_REFERENCE=False` to run your code. `USE_REFERENCE=True` selects all instructor implementations for that level and trains a new model; it does not load a pretrained answer and is not a student submission.

The Challenges cover [Wave](../../02_challenges/01_wave/Challenge_1_Wave_Dynamics.ipynb) (3 levels), [Fluid](../../02_challenges/02_fluid/Challenge_2_Fluid_Flow.ipynb) (3 levels), [Climate](../../02_challenges/03_climate/Challenge_3_Climate_Modeling.ipynb) (2 levels), and [Neural Operators](../../02_challenges/04_neural_operators/Challenge_4_Neural_Operators.ipynb) (3 levels).

[Start Here](../../Start_Here.ipynb) · Previous: [Lab 3: Heat conduction](../03_heat_conduction/Lab_3_Heat_Conduction.ipynb) · Next: [Challenge 1: Wave](../../02_challenges/01_wave/Challenge_1_Wave_Dynamics.ipynb)

--- 

Further reading: [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources). Community support: [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack).

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.